# 리포트 11 — 실측 계획 — 무엇을 재야 이 문서가 닫히나

> 시뮬레이션이 선언으로 남겨 둔 것들의 목록과, 그것을 닫는 **야외 실측 규약**이다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | 가장 보수적인 D 정의로도 세션 거리 하나가 두 기체 세 밴드를 덮는다 | `_parts/69_site-geometry.ipynb` |
| 2 | 구가 σ 를 절대량으로 만들고, 반경 17.8 cm 를 고른다 | `_parts/70_calibration-sphere.ipynb` |
| 3 | 표적을 한 거리빈에 넣는 최대 대역은 200 MHz 다 | `_parts/71_subband.ipynb` |
| 4 | 가장 촘촘한 요구 1.38° 를 세션 간격으로 채택해 앵커 문헌의 고정 2° 보다 촘촘하게 간다 | `_parts/72_attitude.ipynb` |
| 5 | σ(f) 레인지·파형축·비행검출로 층을 나눈다 | `_parts/73_three-layers.ipynb` |
| 6 ⭐ | 캠페인이 결판내는 양은 절대값이 아니라 순위다 | `_parts/74_sim-vs-meas.ipynb` |
| 7 | 기울기 판정의 문턱은 세션간 진폭 재현성이고, σ 사슬 세대를 바꾸면 그 문턱이 손닿는 범위 밖으로 좁아진다 | `_parts/76_session-drift.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열한 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. 가장 보수적인 D 정의로도 세션 거리 하나가 두 기체 세 밴드를 덮는다



> ### 한 일
> **원거리장 2D²/λ 를 D 정의 세 가지로 계산하고, 야외 부지의 지면반사 유령이 거리게이팅으로 떨어지는 기하를 함께 셌다.**

### 결과
1. 채택한 D 는 가장 보수적인 정의다 — 회전 로터 디스크까지 포함한 외접상자의 3D 대각이고, 요구거리 최대는 24.44 m [^1] 다 (matrice4e · WiFi).
2. 같은 기체를 모터-모터 대각으로 재정의하면 요구거리가 3.65 배 [^2] 짧아진다 — 정의를 섞으면 거리가 그만큼 틀린다.
3. 세션 거리는 그 최대값 하나로 잡는다. 한 거리가 기체 2종 × 3밴드를 전부 덮는다.
4. 지면반사 유령은 경로차 2hH/R 이 서브밴드 거리분해능보다 클 때 떨어진다 — 27 개 [^3] 기하 중 200 MHz [^4] 서브밴드에서 분리되는 비율이 78% [^5] 다.
5. ⚠ 이 D 는 Matrice 4E 메쉬의 외접상자에서 나오고, 그 형상은 2026-08-04 [^6] 에 공식 CAD 실측으로 정정됐다 — 표의 값은 **정정 전** 메쉬 기준이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 원거리장 요구거리 | 메쉬 외접상자(회전 로터 디스크 포함)의 3D 대각 D 를 세 밴드 λ 에 넣어 2D²/λ 로 계산 — `benchmark/plan_measurement.py` |
| D 정의 세 가지 | env(로터 디스크 포함 외접상자 3D 대각) · bbox(프로펠러 포함 수평 최대치수) · diag(모터-모터, 앵커 문헌 관례). 채택은 가장 보수적인 env |
| 배경 차감 | 배경은 **지지대를 세운 채로** 재고 **복소수로** 뺀다 |
| 지면반사 | 표적을 경유한 지면반사의 경로차 2hH/R 을 안테나 높이 h · 표적 높이 H · 지상거리 R 의 격자에서 계산해 서브밴드 거리분해능과 견줬다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 원거리장 — 2D²/λ 를 두 기체 × 세 밴드로

채택한 D 는 가장 보수적인 정의다 — 회전 로터 디스크까지 포함한 외접상자의 3D 대각. 같은 기체를 모터-모터 대각으로 재정의하면 요구거리가 3.65 [^2]배 짧아진다. 세 정의를 한 표에 나란히 실어 어느 값을 쓰는지 고정한다.


## 세 정의를 한 표에

| 기체 | 밴드 | λ | D_env | **R_ff(env)** | R_ff(bbox) | R_ff(모터대각) |
|---|---|---|---|---|---|---|
| matrice4e | LTE | 163 mm | 0.839 m | 8.65 m | 4.42 m | 2.37 m |
| matrice4e | 5G | 86 mm | 0.839 m | 16.42 m | 8.38 m | 4.50 m |
| matrice4e | WiFi | 58 mm | 0.839 m | 24.44 m | 12.48 m | 6.69 m |
| mini5pro | LTE | 163 mm | 0.497 m | 3.03 m | 1.74 m | 0.93 m |
| mini5pro | 5G | 86 mm | 0.497 m | 5.76 m | 3.30 m | 1.77 m |
| mini5pro | WiFi | 58 mm | 0.497 m | 8.57 m | 4.91 m | 2.63 m |

출처 [^7]


## 세션 거리 하나로 전부 덮는다

![report06_farfield](../outputs/figures/report06_farfield.png)

**그림 1.** 각 기체와 밴드에서 원거리장에 들어가려면 얼마나 멀어야 하는가?
세션 거리는 최대값 24.44 m [^1] 로 잡는다 — 그 한 거리가 두 기체 세 밴드를 전부 덮는다.

⚠ 이 표와 그림의 D 는 Matrice 4E 메쉬의 외접상자에서 나오고, 그 형상은 2026-08-04 [^6] 에 공식 CAD 실측으로 정정됐다([^8]). 로터 디스크를 포함한 대각이라 변화 폭은 작게 잡히지만, 그 크기는 재계산이 정한다. 형상 정정 자체는 [리포트 2-3 절 1 «메쉬 세우기»](02_3_target-mesh.ipynb) 가 적는다.


## 배경 차감과 지면반사 — 야외 부지를 기하로 다룬다

배경 S_BG 는 **지지대를 세운 채로** 재고 **복소수로** 뺀다.

표적을 경유한 지면반사는 경로차 `2hH/R` 이 서브밴드 거리분해능보다 **클 때** 레인지게이팅으로 떨어진다.


## 어떤 부지가 유령을 밀어내나

![report06_ground_bounce](../outputs/figures/report06_ground_bounce.png)

**그림 2.** 어떤 야외 기하가 지면반사 유령을 표적 거리빈 밖으로 밀어내는가?
27 [^3]개 기하 중 200 MHz [^4] 서브밴드에서 분리되는 비율은 78% [^5] 다. 경로차 범위는 0.30 [^9] ~ 6.00 m [^10] 다.

부지 선정은 그림 2 에서 분해능 선 위에 오는 (h, H, R) 조합으로 한다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| β ≤ 45° 안에서 송수신 분리각별 기하를 같은 방식으로 계산한다 | 바이스태틱 세션의 원거리장 거리와 게이팅 임계가 정해진다 | `benchmark/plan_measurement.py` 확장 |
| σ 사슬을 정정된 메쉬로 다시 돌린 뒤 D 를 다시 읽는다 | 세션 거리가 현재 형상 위에서 확정된다 | [리포트 2-3 절 1 «메쉬 세우기»](02_3_target-mesh.ipynb) |
| 후보 부지에서 (h, H, R) 을 실측하고 경로차를 확인한다 | 레인지게이팅으로 유령을 뗄 수 있는 부지가 확정된다 | 이 편의 그림 2 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 10개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report06_derived.json` | `farfield_adopted.R_ff_max_m` | 24.44 |
| [^2] | `outputs/report06_derived.json` | `farfield_adopted.spread_ratio_max` | 3.652 |
| [^3] | `outputs/report06_derived.json` | `ground_bounce_n_geom` | 27 |
| [^4] | `outputs/report06_derived.json` | `ground_bounce_ref_bw_MHz` | 200 |
| [^5] | `outputs/report06_derived.json` | `ground_bounce_sep_frac_200MHz` | 0.7778 |
| [^6] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^7] | `outputs/report06_derived.json` | `farfield` | (6행 표) |
| [^8] | `outputs/meshfix_applied.json` | `per_drone.matrice4e` | (17항목 묶음) |
| [^9] | `outputs/report06_derived.json` | `ground_bounce_min_m` | 0.3 |
| [^10] | `outputs/report06_derived.json` | `ground_bounce_max_m` | 6 |


---

## 절 2. 구가 σ 를 절대량으로 만들고, 반경 17.8 cm 를 고른다



> ### 한 일
> **정밀 PEC 구를 기준체로 두고 정확 Mie 급수로 기준값을 계산해 두 기체·세 밴드에서 여유가 남는 반경을 골랐다.**

### 결과
1. 채택 반경은 17.8 cm [^11] 다 — 세 밴드 모두 Mie−πr² 편차가 작고 앵커 문헌(Yuan)이 쓴 것과 같은 크기다.
2. 교정구는 두 기체·세 밴드에서 예상 σ 보다 최소 +3.73 dB [^12] 밝다 — 같은 이득 설정으로 둘 다 잡히고, 그래야 두 응답의 비율이 그대로 σ 비율이 된다.
3. 세션 **시작과 끝에 한 번씩** 잰다. 두 값의 차가 그 세션의 드리프트이고, 예산 1.00 dB [^13] 안에 든 세션만 자료로 쓴다.
4. ⭐ 이 구가 캠페인에서 값어치가 가장 크다 — 지금 우리 SBR+PO 커널 출력인 **절대 레벨을 측정에 앵커한다**(생산 모드의 평균 레벨이동 0.00 dB [^14]).
5. 채택 반경에서 우리 정확 Mie σ 는 **πr² 광학 점근** -10.02 dBsm [^15] 대비 밴드에 따라 +0.07 [^16] ~ +0.31 dB [^17] 위에 있다 — 앵커 사슬이 선언한 σ_cal 은 -10.00 dBsm [^18] 이고, 그 규약이 곧 이 광학 점근이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기준체 | 정밀 PEC 구. 구는 방위무관이라 정렬 오차가 σ 에 안 들어간다 |
| 기준값 | 정확 Mie 급수로 계산 — `benchmark/mie_pec_sphere.py:207`, `selfcheck()` 보유. πr² 광학 점근과의 차이는 표의 `Mie−πr²` 열에 dB 로 있다 |
| 여유 정의 | 교정구 정확 Mie σ 에서 기체 예상 σ(앵커 레벨 + L² 크기보정)를 뺀 값 [dB] |
| 드리프트 | 세션 시작과 끝에 같은 구를 재고 두 값의 차를 그 세션의 드리프트로 기록한다 |
| 패턴 교정 | 교정구를 표적과 같은 자리·같은 높이에 놓아 안테나 패턴과 체인 이득을 비율로 소거한다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## σ 를 절대량으로 만드는 장치

**정밀 PEC 구**를 쓴다. 구는 방위무관이라 정렬 오차가 σ 에 안 들어간다.

기준값은 **정확 Mie** 로 쓴다 — πr² 광학 점근과의 차이는 아래 표의 `Mie−πr²` 열에 dB 로 있다. 단일 출처는 `benchmark/mie_pec_sphere.py:207` 이고 자체검증 `selfcheck()` 을 갖고 있다.

세션 **시작과 끝에 한 번씩** 잰다. 두 값의 차가 그 세션의 드리프트 예산이고, 그 차가 목표 정확도 1.00 dB [^13] 안에 들어온 세션만 자료로 쓴다.


## 두 후보 구의 정확 Mie σ 와 여유

| 구 | 밴드 | ka | σ_Mie | Mie−πr² | Matrice 4E 대비 여유 | Mini 5 Pro 대비 여유 |
|---|---|---|---|---|---|---|
| r=17.8 cm | LTE | 6.9 | -9.95 dBsm | +0.07 dB | +4.38 dB | +8.44 dB |
| r=17.8 cm | 5G | 13.1 | -9.71 dBsm | +0.31 dB | +4.27 dB | +8.33 dB |
| r=17.8 cm | WiFi | 19.4 | -9.89 dBsm | +0.13 dB | +3.73 dB | +7.79 dB |
| r=25.0 cm | LTE | 9.7 | -6.47 dBsm | +0.60 dB | +7.86 dB | +11.92 dB |
| r=25.0 cm | 5G | 18.3 | -7.06 dBsm | +0.01 dB | +6.93 dB | +10.98 dB |
| r=25.0 cm | WiFi | 27.3 | -7.14 dBsm | -0.07 dB | +6.49 dB | +10.55 dB |

출처 [^19]


## 어느 반경을 고르나

![report06_calibration](../outputs/figures/report06_calibration.png)

**그림 3.** 어느 반경의 교정구가 세 밴드 모두에서 기체 예상 σ 위에 있는가?
채택 반경은 17.8 cm [^11] 다 — 세 밴드 모두 Mie−πr² 편차가 작고 앵커 문헌(Yuan)이 쓴 것과 같은 크기다.

⚠ 이 여유가 견주는 «기체 예상 σ» 는 `rcs_anchor → sigma_anchor` 사슬에서 오고, 그 사슬은 2026-08-04 [^20] 형상 정정 전 메쉬 위·2026-08-07 10:58:22 [^21] Γ(θ) 각도 모양(기본 켬) 이전 커널 위에 있다 — 두 요인의 방위평균 이동(Γ(θ) 는 +0.08 [^22] ~ +0.10 dB [^23])보다 여유가 +3.73 dB [^12] 로 커서 반경 선택은 버틴다.


## 앵커와 사과-대-사과로 견주려면 규약차를 먼저 되돌린다

앵커 사슬은 같은 반경의 금속구를 σ_cal -10.00 dBsm [^18] 로 선언했다 — πr² 광학값 -10.02 dBsm [^15] 이다[^24]. 두 수의 차는 반올림이고, 규약 자체는 πr² 광학 점근이다.

우리가 정확 Mie 로 교정하면 채택 반경에서 우리 σ 는 **그 πr² 규약 대비** 밴드에 따라 +0.07 [^16] ~ +0.31 dB [^17] 위에 있다 — 앵커와 견줄 때 이 항을 먼저 되돌린다.


## 이 구가 값어치가 가장 큰 이유 두 가지

절대 레벨만 보면 모양을 안 닮은 구도 부피를 맞게 골라 넣으면 우리 메쉬와 같은 자리에 온다 — 그 부피는 결과를 보고 고를 수 있는 값이라, 메쉬 부피로 잡은 구는 도로 우리보다 나쁘다([리포트 3 절 4 «구·상자 대조»](03_anchor.ipynb)).

그리고 우리 세 밴드는 전부 PO 근사의 유효 문턱 아래에 부품을 남긴다([리포트 2 절 5 «PO 무릎»](02_kernel.ipynb)). **계산으로 닫히지 않는 축을 이 구가 닫는다.**


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 교정구를 표적과 같은 자리·같은 높이에서 세션 시작과 끝에 잰다 | 지금 우리 SBR+PO 커널 출력인 절대 레벨이 처음으로 측정에 앵커된다 — 생산 모드의 평균 레벨이동 0.00 dB [^14] 가 측정값으로 대체된다 | `src/sigma_anchor.py` 레벨 앵커 등록 |
| 규약차 항을 되돌린 뒤 앵커 문헌과 σ 를 나란히 놓는다 | 우리 σ 와 앵커 σ 의 비교가 사과-대-사과가 된다 | [리포트 3 절 1 «앵커 모드»](03_anchor.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 14개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^11] | `outputs/report06_derived.json` | `calibration_pick.radius_cm` | 17.8 |
| [^12] | `outputs/report06_derived.json` | `calibration_margin_min_db` | 3.732 |
| [^13] | `outputs/report06_derived.json` | `ranking_validation.drift_budget_db` | 1 |
| [^14] | `outputs/report06_derived.json` | `modes.level_shift_production_abs_max_db` | 0 |
| [^15] | `outputs/report06_derived.json` | `layers.cal_pir2_dbsm` | -10.02 |
| [^16] | `outputs/report06_derived.json` | `layers.mie_shift_min_db` | 0.0723 |
| [^17] | `outputs/report06_derived.json` | `layers.mie_shift_max_db` | 0.3051 |
| [^18] | `outputs/report06_derived.json` | `layers.cal_anchor_declared_dbsm` | -10 |
| [^19] | `outputs/report06_derived.json` | `calibration` | (6행 표) |
| [^20] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^21] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^22] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^23] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |
| [^24] | `outputs/measurement_layers.json` | `calibration_convention_gap.anchor_quote` | A metallic sphere with a radius of 17.8 cm (RCS as sigm… |


---

## 절 3. 표적을 한 거리빈에 넣는 최대 대역은 200 MHz 다



> ### 한 일
> **순시대역을 서브밴드로 쪼개 거리분해능이 기체 최대치수보다 커지는 최대 대역을 두 기체에서 함께 찾았다.**

### 결과
1. 두 기체를 함께 만족시키는 최대 서브밴드는 200 MHz [^25] 다.
2. peak |s|² 를 σ 로 쓰려면 표적이 **한 거리빈 안**에 들어와야 한다 — 조건은 ΔR = c/2B > D_bbox 다.
3. 서브밴드마다 σ 를 내면 그 다발이 곧 σ(f) 이고, 앵커 문헌(Das)의 절차와 같다.
4. 게이팅은 넓게, 평가는 좁게 한다 — 게이팅 400 MHz [^26] 전대역, 평가 50 MHz [^27] 서브밴드 8 개 [^28].
5. 이 대역 상한이 **절 1** «부지 기하» 의 지면반사 분리 조건과 같은 눈금을 쓴다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 점표적 조건 | ΔR = c/2B 가 기체 최대치수 D_bbox 보다 커야 한다 — 그래야 표적이 한 거리빈에 든다 |
| D_bbox | 프로펠러를 포함한 수평 최대치수. 원거리장에서 채택한 env 정의와 구별한다 |
| 게이팅과 평가 | 앵커는 6차 Kaiser 창으로 CIR 을 게이팅한 뒤 주파수축으로 되돌린다[^29]. 우리는 전대역에서 게이팅하고 σ 는 서브밴드로 평가한다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 순시대역을 쪼개 σ(f) 로 만든다

peak |s|² 를 σ 로 쓰려면 표적이 **한 거리빈 안**에 들어와야 한다 (ΔR = c/2B > D_bbox).

두 기체를 함께 만족시키는 최대 서브밴드는 200 MHz [^25] 다. 서브밴드마다 σ 를 내면 그 다발이 곧 σ(f) 이고, 앵커 문헌의 절차와 같다.


## 대역별 점표적 조건

| 대역 B | ΔR = c/2B | Matrice 4E | 여유 | Mini 5 Pro | 여유 |
|---|---|---|---|---|---|
| 400 MHz | 0.375 m | ⚠ 퍼짐 | -0.225 m | ⚠ 퍼짐 | -0.001 m |
| 200 MHz | 0.749 m | 점표적 | +0.150 m | 점표적 | +0.373 m |
| 100 MHz | 1.499 m | 점표적 | +0.900 m | 점표적 | +1.123 m |
| 50 MHz | 2.998 m | 점표적 | +2.399 m | 점표적 | +2.622 m |

출처 [^30]


## 게이팅은 넓게, 평가는 좁게

게이팅(되돌아온 신호에서 필요한 시간 구간만 창으로 잘라내기)은 넓게, 평가는 좁게 한다. 우리는 400 MHz [^26] 전대역에서 게이팅한 뒤 σ 는 50 MHz [^27] 서브밴드 8 [^28] 개로 평가한다.

앵커는 6차 Kaiser 창(가장자리를 부드럽게 깎는 시간창)으로 CIR(채널 임펄스 응답 — 한 번 때린 신호가 되돌아오는 모양)을 게이팅한 뒤 주파수축으로 되돌린다[^29].


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 서브밴드마다 σ 를 내고 그 다발을 σ(f) 로 묶는다 | 우리 σ(f) 가 앵커 문헌과 같은 절차 위에 서고 기울기 판정에 쓸 수 있게 된다 | **절 7** «기울기 판정 문턱» |
| 서브밴드 폭을 바꿔 σ(f) 가 폭에 의존하는지 잰다 | 점표적 가정이 실제로 성립하는 폭이 측정값으로 확정된다 | `benchmark/plan_measurement.py` 확장 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 6개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^25] | `outputs/report06_derived.json` | `point_target_max_bw_MHz` | 200 |
| [^26] | `outputs/report06_derived.json` | `layers.gate_bw_mhz` | 400 |
| [^27] | `outputs/report06_derived.json` | `layers.eval_bw_mhz` | 50 |
| [^28] | `outputs/report06_derived.json` | `layers.n_subbands` | 8 |
| [^29] | `outputs/measurement_layers.json` | `gate_wide_evaluate_narrow.anchor_quote` | a 6th-order Kaiser window function, defined by a predet… |
| [^30] | `outputs/report06_derived.json` | `point_target` | (4행 표) |


---

## 절 4. 가장 촘촘한 요구 1.38° 를 세션 간격으로 채택해 앵커 문헌의 고정 2° 보다 촘촘하게 간다



> ### 한 일
> **방위 각도표본 간격을 밴드마다 λ/4D 로 정하고 앵커 문헌의 고정 간격과 밴드별로 견줬다.**

### 결과
1. 가장 촘촘한 요구는 1.38° [^31] 다 (`matrice4e` · `WiFi`) — 한 바퀴에 262 표본 [^32] 이다.
2. 앵커 문헌은 밴드와 무관하게 2.00° [^33] 고정(반원 91 점 [^34])을 썼고, 우리는 요구를 밴드마다 λ/4D 로 계산해 세션은 최촘값으로 돈다.
3. 우리 요구 표본수는 반원당 28 [^35] ~ 145 점 [^36] 이다.
4. 앵커가 스스로 «높은 주파수에서 성기다» 고 적은 자리는 우리 기체에서 `matrice4e @ WiFi 5.21 GHz` 와 `matrice4e @ ISM 5.8 GHz` 두 칸이다.
5. 로터는 **정지**시키고 블레이드 방위를 기록한다 — 앵커가 회전 성분을 뺐으므로 그 규약에 맞춘다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 표본 간격 | `λ/(4·D_bbox)` [deg] 이하로 잡는다 — 방위 각도표본 권장 간격 |
| 턴테이블 | 엔코더 턴테이블로 방위를 돌리고 각 표본의 각도를 기록한다 |
| 로터 규약 | 로터는 정지시키고 블레이드 방위를 기록한다 — 앵커 문헌이 회전 성분을 뺐다 |
| 앵커 대비 | 밴드별로 우리 요구 간격이 앵커의 고정 간격보다 촘촘한지 표의 마지막 열에 그대로 싣는다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 요구는 λ/4D 로 계산하고, 세션은 최촘값으로 돈다

방위는 엔코더 턴테이블로 돌린다. 요구 간격은 밴드마다 `λ/4D` 로 계산하고, 세션 간격은 그 최촘값 하나로 채택한다 — 칸마다 간격을 바꾸는 대신 전 칸을 최촘 요구로 덮는다.

가장 촘촘한 요구는 1.38° [^31] (`matrice4e` · `WiFi`)이고, 한 바퀴에 262 [^32] 표본이다.

로터는 **정지**시키고 블레이드 방위를 기록한다 — 앵커가 회전 성분을 뺐으므로 그 규약에 맞춘다.


## 밴드별 요구 간격

| 기체 | 밴드 | Δφ 나이퀴스트 | Δφ 권장 | 한 바퀴 표본수 | 앵커 고정 2° 보다 촘촘한가 |
|---|---|---|---|---|---|
| matrice4e | LTE | 7.78° | 3.89° | 93 | 아니오 |
| matrice4e | 5G | 4.09° | 2.05° | 176 | 아니오 |
| matrice4e | WiFi | 2.75° | 1.38° | 262 | 예 |
| mini5pro | LTE | 12.39° | 6.20° | 58 | 아니오 |
| mini5pro | 5G | 6.53° | 3.26° | 110 | 아니오 |
| mini5pro | WiFi | 4.38° | 2.19° | 164 | 아니오 |

출처 [^37]


## 앵커가 성기다고 적은 자리

앵커는 높은 주파수에서 그 고정 간격이 성기다고 스스로 적었고[^38], 우리 기체에서 그 자리는 `matrice4e @ WiFi 5.21 GHz` 와 `matrice4e @ ISM 5.8 GHz` 두 칸이다.

그 둘 중 하나는 2·3층 반송파 ISM 5.8 GHz [^39] 이고, 위 표는 σ(f) 레인지의 세 밴드만 싣는다 — 앵커 대비 판정은 거기에 그 반송파를 더한 칸에서 읽는다.

나머지 칸에서는 앵커의 고정 간격이 우리 요구보다 촘촘하다. 요구 표본수는 반원당 28 [^35] ~ 145 [^36] 점이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 정지 로터 세션과 별도로 회전 세션을 잡는다 | 마이크로도플러가 앵커와의 사과-대-사과를 깨지 않고 들어온다 | [리포트 6-6 절 1 «자세와 가림»](06_6_microdoppler-limits.ipynb) |
| 방위 스윕으로 자세평균 σ 를 내고 로브 위치를 설계값과 대조한다 | 자세 패턴을 기하에서 계산했다는 주장이 결판난다 | [리포트 1 절 3 «결정표»](01_map.ipynb) 의 첫 행 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 9개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^31] | `outputs/report06_derived.json` | `aspect_finest_deg` | 1.375 |
| [^32] | `outputs/report06_derived.json` | `aspect_n_az_max` | 261.7 |
| [^33] | `outputs/report06_derived.json` | `layers.anchor_step_deg` | 2 |
| [^34] | `outputs/report06_derived.json` | `layers.anchor_N` | 91 |
| [^35] | `outputs/report06_derived.json` | `layers.N_required_min` | 28 |
| [^36] | `outputs/report06_derived.json` | `layers.N_required_max` | 145 |
| [^37] | `outputs/report06_derived.json` | `aspect` | (6행 표) |
| [^38] | `outputs/measurement_layers.json` | `angular_sampling._rule` | ⭐ 앵커가 스스로 밝힌 한계(*'the chosen step size may be too large… |
| [^39] | `outputs/report06_derived.json` | `layers.carrier_ism_ghz` | 5.8 |


---

## 절 5. σ(f) 레인지·파형축·비행검출로 층을 나눈다



> ### 한 일
> **캠페인을 세 층으로 나누고 층마다 여는 축과 그 대가를 적었다.**

### 결과
1. 1층은 σ(f) 레인지다 — 정지 표적·턴테이블 방위컷·교정구·배경 코히런트 차감으로 σ(f, φ) 의 분포 P(σ) 를 낸다.
2. 2층은 파형축이다 — 세 파형 구조를 ISM 5.8 GHz [^40] 한 반송파의 같은 서브밴드 중심에 겹쳐 송신하고 SNR_out / E_tx 를 낸다.
3. 3층은 비행 검출이다 — 로터가 도는 유일한 층이고, 고정 Pfa 에서 `Pd vs range at fixed Pfa` 를 낸다.
4. 2층을 한 반송파에 고정하는 이유는 면허다. 잃는 반송파축은 수신전용 검증 3점(LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz [^41])의 실제 배치신호와 v_max 의 λ 비 이전으로 갚는다.
5. 2·3층 ISM 5.8 GHz [^40] 의 원거리장은 채택 정의 env(로터 디스크를 포함한 외접상자 3D 대각)로 27.21 m, bbox(프로펠러를 포함한 수평 최대치수) 정의로 13.69 m [^42] 다 — 2·3층 세션 거리는 그 env 값 이상으로 따로 잡고, σ(f) 레인지 세 밴드의 세션 거리 24.44 m [^43] 는 그 아래에 있다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 층 나누기 | 1층 = σ(f) 레인지, 2층 = ISM 한 반송파의 파형축, 3층 = 비행 검출. 값은 `outputs/measurement_layers.json` 에서 옮기거나 단위만 바꿨다 |
| 한 반송파로 묶는 이유 | 2.1 GHz 야외 송신은 허가가 필요하고 '2.1 GHz 의 WiFi' 라는 배치신호는 세상에 존재하지 않는 인공물이다[^44] |
| 1층이 내는 것 | 점별 패턴이 아니라 **분포 P(σ)** 다 — 검출확률이 σ 분포의 함수이므로 그 분포를 Swerling 틀에 넣어 3층을 예측하고 3층이 그 예측을 검사한다 |
| 수신 채널 | 검증 3점은 수신전용이고 기준 1 + 감시 1 = 2 채널 [^45] 을 같은 클럭에서 쓴다 |
| 2·3층 원거리장 | 원장은 이 반송파의 원거리장을 bbox 정의로만 싣는다 — 채택 정의 env 값은 원장의 D_env 와 λ_ISM 에서 2D²/λ 로 낸다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 세 층

| 층 | 무엇을 재나 | 반송파 | 산출 |
|---|---|---|---|
| 1층 — σ(f) 레인지 [^46] | 정지 표적 · 턴테이블 방위컷 · 교정구 · 배경 코히런트 차감 [^47] | LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz [^41] | σ(f, φ) 의 분포 P(σ) [^48] |
| 2층 — 파형축 [^49] | 세 파형 구조를 같은 서브밴드 중심에 겹쳐 송신 [^50] | ISM 5.8 GHz [^40] 단일 · 150 MHz [^51] 폭 | SNR_out / E_tx [^52] |
| 3층 — 비행 검출 [^53] | 로터가 도는 비행 표적 — 로터가 도는 유일한 층 [^54] | ISM 5.8 GHz [^40] | 고정 Pfa 에서 Pd(range) [^55] |


## 2층을 한 반송파에 고정하는 이유

면허다 — 2.1 GHz 야외 송신은 허가가 필요하고 '2.1 GHz 의 WiFi' 라는 배치신호는 세상에 존재하지 않는 인공물이다[^44].

잃는 반송파축은 수신전용 검증 3점(LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz [^41])의 실제 배치신호와 v_max 의 λ 비 이전으로 갚는다 — 그 3점은 교차설계의 대각선이 아니라 독립 검사점이다.


## 1층이 내는 것은 분포다

1층이 내는 것은 점별 패턴이 아니라 **분포 P(σ)** 다. 검출확률이 σ 분포의 함수이므로, 그 분포를 Swerling 틀(표적 밝기가 얼마나 요동하는지를 몇 가지 표준 분포로 나눈 레이다 관례 분류)에 넣어 3층의 `Pd vs range at fixed Pfa` 를 예측하고 3층이 그 예측을 검사한다[^56].


## 2·3층의 원거리장 — 채택한 D 정의로 잰다

2·3층은 ISM 5.8 GHz [^40] 한 반송파에서 돈다. 그 반송파의 원거리장을 두 D 정의로 나란히 싣는다 — 기준 기체는 요구가 가장 큰 `matrice4e` 다.

| D 정의 | D | 2D²/λ |
|---|---|---|
| env — 로터 디스크를 포함한 외접상자의 3D 대각 (**절 1** «부지 기하» 채택) | 0.839 m [^57] | 27.21 m |
| bbox — 프로펠러를 포함한 수평 최대치수 | 0.595 m [^58] | 13.69 m [^42] |

채택 정의로 2·3층 세션 거리는 27.21 m 이상이다 — **절 1** «부지 기하» 가 σ(f) 레인지 세 밴드에서 잡은 24.44 m [^43] 는 그 아래에 있고, 2·3층은 자기 거리를 그 위에서 따로 잡는다.

⚠ 원장은 이 반송파의 원거리장을 bbox 정의로만 싣는다([^42]). 위 env 값은 원장의 D_env 와 λ_ISM 0.0517 m [^59] 에서 2D²/λ 로 낸 것이다.

⚠ 위 두 D 는 Matrice 4E 메쉬의 외접상자에서 나오고, 그 형상은 2026-08-04 [^60] 에 공식 CAD 실측으로 정정됐다 — 표의 값은 **정정 전** 메쉬 기준이라 **절 1** «부지 기하» 의 표와 같은 계열이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 자세축과 로터위상축을 σ 생산자에 배선한 뒤 sim-to-sim ablation 을 돌린다 | 우리 σ 가 (σ̄, τ_decorr, 분포형) 3개 수로 환원되는지가 실측 없이 판정된다 — 태스크는 분류가 아니라 **검출**로 고정한다 | `docs/SIM2REAL_PLAN.md` → `outputs/s2r_protocol.json` |
| 3팔 실측 ablation 설계를 검출 태스크와 supervision 사다리로 다시 짠다 | 현 설계 판정 RISKY [^61] 의 근거가 닫힌다 — 자세축·로터위상축을 배선하기 전에는 세 팔이 분포형·상관시간 두 수로 환원되어 판정이 설계로 보장된다 | `outputs/s2r_attack.json` → `docs/SIM2REAL_PLAN.md` |
| 1층의 P(σ) 를 Swerling 틀에 넣어 3층의 Pd(range) 를 예측한다 | 1층과 3층이 하나의 예측-검사 고리로 묶인다 | `outputs/measurement_layers.json : layer3_flight.ties_back_to` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 22개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^40] | `outputs/report06_derived.json` | `layers.carrier_ism_ghz` | 5.8 |
| [^41] | `outputs/report06_derived.json` | `layers.validation_points` | LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz |
| [^42] | `outputs/report06_derived.json` | `layers.farfield_ism_bbox_max_m` | 13.69 |
| [^43] | `outputs/report06_derived.json` | `farfield_adopted.R_ff_max_m` | 24.44 |
| [^44] | `outputs/measurement_layers.json` | `layer2_waveform_axis.why_one_carrier` | 면허. 2.1 GHz 야외 송신은 허가가 필요하고 '2.1 GHz 의 WiFi' 는 세상에 없는 인… |
| [^45] | `outputs/report06_derived.json` | `layers.n_channels` | 2 |
| [^46] | `outputs/report06_derived.json` | `layers.rows[0].layer` | 1층 — σ(f) 레인지 |
| [^47] | `outputs/report06_derived.json` | `layers.rows[0].measures` | 정지 표적 · 턴테이블 방위컷 · 교정구 · 배경 코히런트 차감 |
| [^48] | `outputs/report06_derived.json` | `layers.rows[0].product` | σ(f, φ) 의 분포 P(σ) |
| [^49] | `outputs/report06_derived.json` | `layers.rows[1].layer` | 2층 — 파형축 |
| [^50] | `outputs/report06_derived.json` | `layers.rows[1].measures` | 세 파형 구조를 같은 서브밴드 중심에 겹쳐 송신 |
| [^51] | `outputs/report06_derived.json` | `layers.span_ism_mhz` | 150 |
| [^52] | `outputs/report06_derived.json` | `layers.rows[1].product` | SNR_out / E_tx |
| [^53] | `outputs/report06_derived.json` | `layers.rows[2].layer` | 3층 — 비행 검출 |
| [^54] | `outputs/report06_derived.json` | `layers.rows[2].measures` | 로터가 도는 비행 표적 — 로터가 도는 유일한 층 |
| [^55] | `outputs/report06_derived.json` | `layers.rows[2].product` | 고정 Pfa 에서 Pd(range) |
| [^56] | `outputs/measurement_layers.json` | `layer3_flight.ties_back_to` | 1층의 P(sigma) 를 Swerling 틀로 넣으면 Pd(range) 가 예측된다. 3층은 그… |
| [^57] | `outputs/report06_derived.json` | `farfield[0].D_env_m` | 0.8385 |
| [^58] | `outputs/measurement_layers.json` | `layer1_at_ism.airframes.matrice4e.D_bbox_m` | 0.5947 |
| [^59] | `outputs/measurement_layers.json` | `layer1_at_ism.lambda_m` | 0.05169 |
| [^60] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^61] | `outputs/s2r_attack.json` | `verdict` | RISKY |


---

## 절 6. 캠페인이 결판내는 양은 절대값이 아니라 순위다



> ### 한 일
> **자세평균 σ 위에서 파형 순위를 뒤집는 데 필요한 밴드별 σ 이동폭을 캠페인의 진폭 재현성 요구치로 삼고 세션 드리프트 예산과 견줬다.**

### 결과
1. 기체 5 종 [^62]이 순위 `L1 > G1 > W1` 에 합의한다.
2. 그 순위를 뒤집는 밴드별 σ 이동폭은 Matrice 4E 1.30 dB [^63] · Mini 5 Pro 2.95 dB [^64] 다.
3. 세션 드리프트 예산 1.00 dB [^65] 가 그 폭 아래에 있다 — 여유 +0.30 dB [^66]. 그 여유는 결정론적 값이고, 같은 예산을 밴드별 독립 오차로 놓은 단일자세 몬테카를로에서 순위 보존확률은 Matrice 4E 0.583 [^67] · Mini 5 Pro 0.992 [^68] 다.
4. σ 를 세 밴드 공통으로 1 dB 옮기면 절대거리가 0.25 dB [^69] 움직인다 — 순위는 그 공통이동에서 불변이다.
5. ⚠ 그 폭을 정하는 것은 Matrice 4E 행이고, 그 행은 2026-08-04 [^70] 형상 정정 전 메쉬·2026-08-07 10:58:22 [^71] Γ(θ) 이전 커널 값이다 — σ 사슬을 두 축 한 라운드로 다시 돌려 그 행이 여유만큼만 움직여도 이 «맞는다» 판정이 뒤집힌다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 판정 대상 | 자세평균 σ 위의 파형 순위 뒤집힘 폭을 캠페인 요구치로 삼음 — `benchmark/sigma_sensitivity.py:470` |
| 왜 순위인가 | 순위를 정하는 λ²·점유·대역폭 항은 밴드 간 차이고 환경 항은 세 밴드 공통이다 — 야외 환경이 시뮬 자유공간과 달라도 밴드 간 차는 남는다 |
| 몬테카를로 기저 | 단일자세 lead 위의 몬테카를로 — `benchmark/sigma_sensitivity.py:430`. 캠페인은 방위 1.38° [^72] 표본으로 자세평균 σ 를 낸다 |
| 절대 σ 는 다음 라운드 | [리포트 10-2 절 5 «체크리스트»](10_2_robustness.ipynb) 여섯 항목을 다 채운 세션이 만든다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 캠페인이 결판내는 양

자세평균 σ 에서 기체 5 종 [^62]이 순위 `L1 > G1 > W1` 에 합의하고, 그 순위를 뒤집는 밴드별 σ 이동폭은 Matrice 4E 1.30 dB [^63] · Mini 5 Pro 2.95 dB [^64] 다.

세션 드리프트 예산 1.00 dB [^65] 가 그 폭 아래에 있으므로(여유 +0.30 dB [^66]) 이 캠페인은 **검출 사슬과 파형 순위**를 결판낸다.


## 그 여유를 확률로 다시 읽으면

위 여유는 뒤집힘 폭 하나와 예산 하나를 견준 결정론적 값이다. 같은 예산 1.00 dB [^65] 를 밴드별 독립 오차로 놓고 **단일자세 lead** 위에서 몬테카를로를 돌리면 순위 보존확률은 Matrice 4E 0.583 [^67] · Mini 5 Pro 0.992 [^68] 다[^73].

그러므로 세션 설계가 노리는 것은 예산을 겨우 지키는 것이 아니라 뒤집힘 폭보다 한참 아래로 내리는 것이다. 기체별 확률 전량은 [리포트 3 절 6 «σ 강건성»](03_anchor.ipynb) 의 표에 있다.


## 설계상 같게 맞춘 축과 다르게 둔 축

| 설계상 같게 맞춘 축 | 설계상 다르게 둔 축 |
|---|---|
| 바이스태틱 구조 — 기준 1 + 감시 1, 공통 클럭 | 환경 — 시뮬은 자유공간, 실측은 지면반사·다중경로 |
| 같은 기체 2종 (Matrice 4E · Mini 5 Pro) | 클러터 — 실측 부지의 정적 산란체 |
| 같은 세 파형 (LTE · 5G · WiFi) | 동적범위 — 시뮬 ECA 는 float64, 실측은 12-bit |
| 같은 검출기 사슬 (ECA → 거리도플러 → CA-CFAR) | 자세 — 시뮬은 각도격자, 비행 중에는 자유 |
| σ 통계 규약 (방위 선형평균) | 링크예산 전제 — 절대 탐지거리 |


## 앵커 원장 — 이 캠페인이 닫으러 가는 항목

| 항목 | 상태 | 크기 |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 dB |
| size transfer law | UNRESOLVED | +9.50 dB |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 dB |
| near-field vs far-field, environment | OK | +0.00 dB |

출처 [^74]


## 재보정 모드 — 어느 숫자가 어느 모드에서 오나

생산 σ 원장은 `slope_only [^75]` 다 — 주파수 기울기만 측정에서 받고 **절대 레벨은 우리 SBR+PO 커널 출력 그대로**다(평균 레벨이동 0.00 dB [^76]). 레벨을 앵커에 맞추는 두 모드는 설계 계산에만 쓴다.

| 모드 | 무엇을 옮기나 | Matrice 4E | Mini 5 Pro |
|---|---|---|---|
| slope_only (생산 기본) | 주파수 기울기만 | +0.00 dB | +0.00 dB |
| level_and_slope_L2 | 레벨 + 기울기 (L²) | +6.47 dB | +4.03 dB |
| level_and_slope_L4 | 레벨 + 기울기 (L⁴) | +8.44 dB | +1.93 dB |

출처 [^77]


## 두 기체가 앵커에 대해 어디 서 있나

| 기체 | 앵커 대비 등급 | 크기비 | L² 보정 | L⁴ 보정 |
|---|---|---|---|---|
| Matrice 4E | scaled | 1.254 [^78] | +1.96 dB [^79] | +3.93 dB [^80] |
| Mini 5 Pro | scaled | 0.786 [^81] | -2.09 dB [^82] | -4.19 dB [^83] |

두 기체 다 등급이 `scaled` 다 — 앵커 기체와 같은 4로터 위상이고 대각만 다르다.

⚠ 위 표들은 `rcs_anchor → sigma_anchor` 사슬 위에 서 있고, 그 사슬은 2026-08-04 [^70] 형상 정정 **전** 메쉬·2026-08-07 10:58:22 [^71] Γ(θ) 각도 모양(기본 켬) **이전** 커널로 돌린 것이다 — 다섯 앵커 기체 중 Matrice 4E 가 그 정정을 받았다([^84]).


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 야외 고정기하에서 세 파형을 같은 세션에 송신하고 방위 스윕으로 자세평균 순서를 잰다 | 파형 상대순위 `L1 > G1 > W1` 가 실측에서 확인된다 — 판정 문턱은 뒤집힘 폭 1.30 dB [^85] 다 | [리포트 1 절 3 «결정표»](01_map.ipynb) → 검출 결과 편과 대조 |
| σ 사슬을 정정된 메쉬·Γ(θ) 켠 커널 한 라운드로 다시 돌린 뒤 뒤집힘 폭을 다시 읽는다 | 설계와 판정 대상의 «맞는다» 가 현재 형상·현재 커널 위에서 확정된다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 24개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^62] | `outputs/report06_derived.json` | `ranking_validation.n_drones` | 5 |
| [^63] | `outputs/report06_derived.json` | `ranking_validation.flip_span_db.matrice4e` | 1.298 |
| [^64] | `outputs/report06_derived.json` | `ranking_validation.flip_span_db.mini5pro` | 2.946 |
| [^65] | `outputs/report06_derived.json` | `ranking_validation.drift_budget_db` | 1 |
| [^66] | `outputs/report06_derived.json` | `ranking_validation.drift_margin_db` | 0.2979 |
| [^67] | `outputs/report06_derived.json` | `ranking_validation.p_order_preserved_at_1db.matrice4e` | 0.5828 |
| [^68] | `outputs/report06_derived.json` | `ranking_validation.p_order_preserved_at_1db.mini5pro` | 0.9925 |
| [^69] | `outputs/report06_derived.json` | `ranking_validation.common_mode_slope_db_per_db` | 0.2462 |
| [^70] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^71] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^72] | `outputs/report06_derived.json` | `ranking_validation.az_step_deg` | 1.375 |
| [^73] | `outputs/report06_derived.json` | `ranking_validation.mc_basis` | 단일자세 lead 위의 몬테카를로 (benchmark/sigma_sensitivity.py:430) |
| [^74] | `outputs/sigma_anchor.json` | `uncontrolled` | (6행 표) |
| [^75] | `outputs/report06_derived.json` | `modes.production_mode` | slope_only |
| [^76] | `outputs/report06_derived.json` | `modes.level_shift_production_abs_max_db` | 0 |
| [^77] | `outputs/report06_derived.json` | `modes.rows` | (3행 표) |
| [^78] | `outputs/sigma_anchor.json` | `drones.matrice4e.comparability.size_ratio` | 1.254 |
| [^79] | `outputs/sigma_anchor.json` | `drones.matrice4e.comparability.size_corr_L2_db` | 1.964 |
| [^80] | `outputs/sigma_anchor.json` | `drones.matrice4e.comparability.size_corr_L4_db` | 3.928 |
| [^81] | `outputs/sigma_anchor.json` | `drones.mini5pro.comparability.size_ratio` | 0.7857 |
| [^82] | `outputs/sigma_anchor.json` | `drones.mini5pro.comparability.size_corr_L2_db` | -2.095 |
| [^83] | `outputs/sigma_anchor.json` | `drones.mini5pro.comparability.size_corr_L4_db` | -4.189 |
| [^84] | `outputs/meshfix_attack.json` | `Q6_invalidated_outputs.critical[1]` | (3항목 묶음) |
| [^85] | `outputs/report06_derived.json` | `ranking_validation.flip_span_min_db` | 1.298 |


---

## 절 7. 기울기 판정의 문턱은 세션간 진폭 재현성이고, σ 사슬 세대를 바꾸면 그 문턱이 손닿는 범위 밖으로 좁아진다



> ### 한 일
> **우리 커널의 밴드 기울기와 앵커 두 개의 기울기가 대역 끝에서 벌리는 간격을 계산해 세션 재현성 요구치로 못 박고, 세대를 바꿔 같은 계산을 다시 했다.**

### 결과
1. 판정 문턱은 1.77 dB [^86] 다 — 앵커 두 개(전대역 Das · 창을 맞춘 저대역 Yuan) 중 **좁은 쪽**이다.
2. 우리 커널은 0.936 [^87] ~ 1.517 [^88] dB/GHz 이고, 앵커는 전대역 0.210 [^89] · 창을 맞춘 쪽 0.411 [^90] dB/GHz 다.
3. 대역 3.367 GHz [^91] 를 지나며 두 가설이 벌리는 폭은 전대역 앵커에서 2.44 [^92] ~ 4.40 [^93] dB, 창을 맞춘 앵커에서 1.77 [^94] ~ 3.73 [^95] dB 다.
4. ⚠⚠ 이 문턱은 생산 원장 세대(2026-07-30 07:16:08 [^96])의 수다. 디스크의 현재 세대(2026-08-03 05:49:19 [^97])로 같은 정의를 다시 적합하면 `matrice4e` 가 0.936 [^87] → 0.233 dB/GHz [^98] 로 내려간다.
5. 그 세대에서 가장 좁은 판별폭은 0.08 dB [^99] 로, 세션 진폭 재현성이 닿는 범위 밖이다 — 이 행을 «결판» 으로 유지하려면 σ 사슬 재실행이 선행 조건이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기울기 정의 | 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0). 정의는 하나로 고정한다 — 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) [^100] |
| 판별폭 | gap = (ours − anchor) × span — 대역 끝에서 두 가설이 벌리는 간격 [dB]. 세션간 진폭 재현성이 이보다 좋아야 기울기 판정이 성립한다 |
| 앵커 두 개 | 전대역 적합값(Das)과 이 캠페인에 창을 맞춘 저대역 적합값(Yuan θ=90 복원 실측곡선)을 나란히 쓴다. 문턱은 두 앵커가 주는 판별폭 중 **좁은 쪽**을 세션 재현성 요구로 쓴다 [^101] |
| 두 세대 | 생산 원장은 한 세대 앞선 `rcs_anchor.json` 위에 서 있다. 디스크의 현재 판으로 같은 정의를 다시 적합한 값을 나란히 싣는다 — 문턱은 생산 원장 값으로 그대로 둔다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---


## 세션 재현성이 판정의 문턱이다

![report06_slope](../outputs/figures/report06_slope.png)

**그림 4.** 우리 기울기와 앵커 기울기를 가르려면 세션 재현성이 얼마나 좋아야 하는가?
우리 커널은 0.936 [^87] ~ 1.517 [^88] dB/GHz 다. 앵커는 **두 개를 나란히** 쓴다.


## 두 앵커와 두 창

| 앵커 | 기울기 [dB/GHz] | 적합 창 [GHz] |
|---|---|---|
| 전대역 (Das) | 0.210 [^89] | 1.8 [^102] ~ 18.2 [^103] |
| 창을 맞춘 저대역 (Yuan theta=90 복원 실측곡선 (EuCAP 2025) [^104]) | 0.411 [^90] | 1.8 [^105] ~ 6.0 [^106] |
| 이 캠페인의 창 | — | 1.843 [^107] ~ 5.21 [^108] |

⚠ 두 창은 **같지 않고 겹친다** — 앵커 창이 이 캠페인의 창을 덮되 위쪽이 더 넓다. 전대역 앵커와 견주면 훨씬 가깝다는 뜻이지 같은 창이라는 뜻이 아니다.


## 판별폭과 문턱

대역 3.367 GHz [^91] 를 지나며 두 가설이 벌리는 폭은 전대역 앵커에서 2.44 [^92] ~ 4.40 [^93] dB, 창을 맞춘 앵커에서 1.77 [^94] ~ 3.73 [^95] dB 다.

**판정 문턱은 둘 중 좁은 쪽 1.77 dB [^86] 로 잡는다** — 창을 맞춘 앵커가 우리 값에 더 가까워서 요구 재현성이 그만큼 빡빡하다([^109]).


## 세대를 바꾸면 문턱이 손닿는 범위 밖으로 간다

위의 «우리 커널» 값은 생산 원장(`sigma_anchor.json`, 2026-07-30 07:16:08 [^96] 판 `rcs_anchor.json` 위)에서 왔다. 디스크의 현재 `rcs_anchor.json`(2026-08-03 05:49:19 [^97] 판)으로 같은 정의를 다시 적합하면 `matrice4e` 가 0.936 [^87] → 0.233 dB/GHz [^98] 로 내려가고, 전대역 앵커와의 판별폭이 2.44 [^110] → 0.08 dB [^111] 가 된다.

그 세대에서 가장 좁은 판별폭은 0.08 dB [^99] 로, 세션 진폭 재현성이 닿는 범위 밖이다. 두 세대 모두 2026-08-04 [^112] 형상 정정 전 메쉬 위·2026-08-07 10:58:22 [^113] Γ(θ) 각도 모양(기본 켬) 이전 커널 위에 서 있고, 앵커 5기체 중 Matrice 4E 가 그 정정을 받았다 — 재실행 선행조건은 형상 정정 + Γ(θ) 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| σ 사슬(rcs_anchor → sigma_anchor)을 형상 정정 + Γ(θ) 한 세대로 다시 돌린 뒤 이 문턱을 다시 낸다 | 기울기 판별폭이 현재 기하 위에서 확정된다 — 지금은 세대를 바꾸는 것만으로 2.44 [^110] → 0.08 dB [^111] 움직인다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |
| 세 밴드를 같은 세션에서 재고 세션간 진폭 재현성을 기록한다 | 밴드 기울기가 우리 커널 값과 앵커 0.210 [^89] dB/GHz 중 어디에 앉는지 결정된다 — 위 사슬 재실행이 이 판정의 선행 조건이다 | [리포트 3 절 1 «앵커 모드»](03_anchor.ipynb) 재기술 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 28개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^86] | `outputs/report06_derived.json` | `slope.threshold_db` | 1.768 |
| [^87] | `outputs/report06_derived.json` | `slope.rows[0].ours_db_per_ghz` | 0.9359 |
| [^88] | `outputs/report06_derived.json` | `slope.rows[1].ours_db_per_ghz` | 1.517 |
| [^89] | `outputs/report06_derived.json` | `slope.anchor_db_per_ghz` | 0.21 |
| [^90] | `outputs/report06_derived.json` | `slope.anchor_band_matched_db_per_ghz` | 0.4107 |
| [^91] | `outputs/report06_derived.json` | `slope.span_ghz` | 3.367 |
| [^92] | `outputs/report06_derived.json` | `slope.gap_db_min` | 2.444 |
| [^93] | `outputs/report06_derived.json` | `slope.rows[1].gap_db` | 4.401 |
| [^94] | `outputs/report06_derived.json` | `slope.gap_db_min_band_matched` | 1.768 |
| [^95] | `outputs/report06_derived.json` | `slope.rows[1].gap_db_band_matched` | 3.725 |
| [^96] | `outputs/report06_derived.json` | `slope.ledger_generation` | 2026-07-30 07:16:08 |
| [^97] | `outputs/report06_derived.json` | `slope.current_generation` | 2026-08-03 05:49:19 |
| [^98] | `outputs/report06_derived.json` | `slope.rows[0].ours_current_generation_db_per_ghz` | 0.2332 |
| [^99] | `outputs/report06_derived.json` | `slope.discrimination_min_abs_db_current_generation` | 0.07819 |
| [^100] | `outputs/report06_derived.json` | `slope.fit_note_short` | 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) |
| [^101] | `outputs/report06_derived.json` | `slope.threshold_rule` | 두 앵커가 주는 판별폭 중 **좁은 쪽**을 세션 재현성 요구로 쓴다 |
| [^102] | `outputs/report06_derived.json` | `slope.anchor_window_ghz[0]` | 1.8 |
| [^103] | `outputs/report06_derived.json` | `slope.anchor_window_ghz[1]` | 18.2 |
| [^104] | `outputs/report06_derived.json` | `slope.anchor_band_matched_source` | Yuan theta=90 복원 실측곡선 (EuCAP 2025) |
| [^105] | `outputs/report06_derived.json` | `slope.anchor_band_matched_window_ghz[0]` | 1.8 |
| [^106] | `outputs/report06_derived.json` | `slope.anchor_band_matched_window_ghz[1]` | 6 |
| [^107] | `outputs/report06_derived.json` | `slope.campaign_window_ghz[0]` | 1.843 |
| [^108] | `outputs/report06_derived.json` | `slope.campaign_window_ghz[1]` | 5.21 |
| [^109] | `outputs/p3_validation_v2.json` | `our_operating_band` | (9항목 묶음) |
| [^110] | `outputs/report06_derived.json` | `slope.rows[0].gap_db` | 2.444 |
| [^111] | `outputs/report06_derived.json` | `slope.rows[0].gap_db_current_generation` | 0.07819 |
| [^112] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^113] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
